In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Cookbook_Building_Complex_Dimers)=
# Building Complex Dimers

*Harvesting, aligning, and merging monomers from disparate PDB structures into a complete complex.*

Biological assemblies deposited in the Protein Data Bank often present incomplete asymmetric units, missing loops, or alternate conformations split across different chains. For instance, PDB entry [1BRS](https://www.rcsb.org/structure/1BRS) contains three Barnase-Barstar heterodimers in the asymmetric unit, but individual chains vary in resolution and sequence completeness.

In this recipe, we demonstrate how to harvest high-quality Barnase and Barstar monomers, align them onto the canonical binding orientation using sequence-aware structural alignment, merge them into a single molecular system, and repair missing heavy atoms and hydrogens.

:::{versionadded} 1.0.0
:::


## Harvesting Monomers

We begin by loading the multi-chain assembly from PDB entry `1BRS`. We strip crystallographic waters and ions to retain purely the protein chains, then extract Barnase (chain B), the best-resolved Barstar monomer (chain F), and the reference bound Barstar partner (chain E):

In [2]:
import molsysmt as msm

# Load the multi-chain assembly and strip solvent/ions
molsys = msm.convert('pdb_id:1BRS')
molsys = msm.extract(molsys, selection="molecule_type=='protein'")

# Inspect the protein chains available in the asymmetric unit
print(msm.get(molsys, element='chain', name=True, n_atoms=True))

# Extract individual monomer chains
barnase = msm.extract(molsys, selection="chain_name=='B'")
barstar_ref = msm.extract(molsys, selection="chain_name=='E'")  # Reference binding pose
barstar_best = msm.extract(molsys, selection="chain_name=='F'") # High-resolution monomer

print(f"Barnase B atoms: {msm.get(barnase, n_atoms=True)}")
print(f"Barstar F atoms: {msm.get(barstar_best, n_atoms=True)}")

[['A', 'B', 'C', 'D', 'E', 'F'], [864, 878, 839, 693, 665, 699]]


Barnase B atoms: 878
Barstar F atoms: 699


## Viewing Monomers

We visualize the harvested Barnase monomer prior to superposition:

In [3]:
molsysviewer_htmlfile = '_static/views/cookbook_dimers_initial.html'


In [4]:
msm.view(barnase)

'<iframe src="../../../_static/views/cookbook_dimers_initial.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

## Structural Alignment

To place the high-resolution Barstar monomer (`barstar_best`) into the functional binding pocket of Barnase, we align its C-alpha backbone onto the bound reference chain (`barstar_ref`) using {func}`molsysmt.structure.least_rmsd_align`:

In [5]:
# Align Barstar F onto the bound coordinates of Barstar E
barstar_aligned = msm.structure.least_rmsd_align(
    barstar_best,
    selection='atom_name=="CA"',
    reference_molecular_system=barstar_ref
)

print("Barstar F coordinates successfully superimposed onto chain E reference pose.")

Barstar F coordinates successfully superimposed onto chain E reference pose.


## Viewing Alignment

We can inspect both monomers in 3D space to verify that Barstar is correctly oriented relative to Barnase:

In [6]:
molsysviewer_htmlfile = '_static/views/cookbook_dimers_aligned.html'


In [7]:
msm.view([barnase, barstar_aligned])

'<iframe src="../../../_static/views/cookbook_dimers_aligned.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

## Merging Monomers

Now that both monomers reside in their relative spatial coordinates, we merge them into a single molecular system with {func}`molsysmt.basic.merge`:

In [8]:
# Merge Barnase and aligned Barstar into a unified dimer
dimer = msm.merge([barnase, barstar_aligned])
msm.info(dimer)

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_proteins,n_structures
molsysmt.MolSys,1577,199,2,2,2,2,2,1


## Chemical Curation

We complete missing heavy atoms and add hydrogen atoms at physiological pH (7.4):

In [9]:
# Repair missing heavy atoms and protonate at physiological pH
dimer = msm.build.add_missing_heavy_atoms(dimer)
dimer = msm.build.add_missing_hydrogens(dimer, pH=7.4)

print(f"Curated Dimer contains {msm.get(dimer, n_atoms=True)} atoms.")

Curated Dimer contains 3159 atoms.


## Viewing Dimer

We visualize the fully assembled and curated Barnase-Barstar complex:

In [10]:
molsysviewer_htmlfile = '_static/views/cookbook_dimers_final.html'


In [11]:
msm.view(dimer)

'<iframe src="../../../_static/views/cookbook_dimers_final.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

## Summary

We have successfully assembled a clean, complete, and repaired Barnase-Barstar heterodimer from separate experimental chains.

In [12]:
# Print final summary metrics
n_atoms, n_groups, n_chains = msm.get(dimer, n_atoms=True, n_groups=True, n_chains=True)
print(f"Constructed Dimer: {n_atoms} atoms across {n_groups} residues in {n_chains} chains.")

Constructed Dimer: 3159 atoms across 199 residues in 2 chains.


:::{seealso}
:class: dropdown

- {func}`molsysmt.structure.least_rmsd_align`: Superimposing structures with sequence-aware least-RMSD alignment.
- {func}`molsysmt.basic.merge`: Merging multiple molecular systems into a unified system.
- {func}`molsysmt.build.add_missing_heavy_atoms`: Repairing missing heavy atoms in protein backbones and side chains.
- {func}`molsysmt.build.add_missing_hydrogens`: Protonating molecular structures at target pH values.
:::